In [15]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.mixture import unifac
from src.data import load_unifac_data, check_unifac_parameters, check_pure_gc_parameters
import numpy as np
import matplotlib.pyplot as plt

In [16]:
# step1: define the compounds
mixture = {#'acetone': {'CH3':1, 'CH3CO':1},
           #'methanol': {'CH3OH':1},
           #'water': {'H2O':1},
           #'DOH': {'DOH':1},
           #'methyl_acetate': {'CH3':1, 'CH3COO':1},
           #'water': {'H2O':1},
           'DMSO': {'DMSO':1},
           #'E': {'CH3':2, 'CH2':1, 'C':1, 'CH3CO':2}
           #'E': {'CH3':4, 'C':1, 'CH2COO':1}
           'heptane': {'CH3':2, 'CH2':5}
           }
# validate the structures
validator = StructuralValidator()
validator.validate(mixture)

{'DMSO': True, 'heptane': True}

In [17]:
# step2: calculate the required properties
prop_list = ['tc', 'pc', 'tb', 'ac', 'hv']
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)
# fix the properties of acetone
# pure_props['acetone']['tc'] = 508.1
# pure_props['acetone']['pc'] = 48.0
# pure_props['acetone']['tb'] = 329.3
# pure_props['methyl_acetate']['tc'] = 510.0     # K
# pure_props['methyl_acetate']['pc'] = 47.5      # bar
# pure_props['methyl_acetate']['tb'] = 330.0
# pure_props['water']['tc'] = 647.1     # K
# pure_props['water']['pc'] = 220.64    # bar
# pure_props['water']['tb'] = 373.15
pure_props

{'DMSO': {'tc': 729.0,
  'pc': 56.49999999999995,
  'tb': 463.1000000000001,
  'ac': 0.28055099999999883,
  'hv': 52.465},
 'heptane': {'tc': 549.8559805147336,
  'pc': 28.26756251167203,
  'tb': 379.0732115086755,
  'ac': 0.3368237739045966,
  'hv': 38.311}}

In [18]:
# check if unifac dat are available
check_unifac_parameters(mixture)
# Load all UNIFAC parameters
rk_data, qk_data, interaction_data = load_unifac_data()

In [19]:
# step4: perform the LLE

# define number of data points
n_data = 100
# set composition
# generate all binary combinations for ternary system
x0 = 0.0
xf = 1.0

# Binary mixture: x1-x2
x1 = np.linspace(x0, xf, n_data)
x2 = 1 - x1

# Composition array
x = np.column_stack([x1, x2])

# generate temperature matrix
tb_values = [pure_props[compound]['tb'] for compound in mixture.keys()]

T_step = 1.0
T_range = np.arange(min(tb_values), max(tb_values), T_step)
T = np.tile(T_range[:, np.newaxis], (1, n_data))

In [20]:
miscibility_results = []
for i in range(T.shape[0]):
    results = unifac(mixture, x, T[i,:], rk_data, qk_data, interaction_data)
    gamma = results['gamma']
    # Gibbs mixing energy
    G_mix = (np.sum(x * np.log(gamma), axis=1) + 
         np.sum(np.ma.log(x).filled(0) * x, axis=1))
    
    # Second derivative test
    G2_mix = np.diff(np.diff(G_mix))
    
    # Miscibility checks (excluding endpoints with [1:-1])
    is_miscible = np.all(G_mix[1:-1] <= 0) and np.all(G2_mix >= 0)
    
    miscibility_results.append({
        'T': T,
        'miscible': is_miscible,
        'G_mix': G_mix,
        'G2_mix': G2_mix
    })
    #print(f"T = {T[i,0]:.2f} K: Miscible = {is_miscible}")
    if not is_miscible:
        print(f"  T = {T[i,0]:.2f} K:")
        #print(f"  - G_mix all <= 0: {np.all(G_mix[1:-1] <= 0)}")
        #print(f"  - G2_mix all >= 0: {np.all(G2_mix >= 0)}")

  T = 379.07 K:
  T = 380.07 K:
  T = 381.07 K:
  T = 382.07 K:
  T = 383.07 K:
  T = 384.07 K:
  T = 385.07 K:
  T = 386.07 K:
  T = 387.07 K:
  T = 388.07 K:
  T = 389.07 K:
  T = 390.07 K:
  T = 391.07 K:
  T = 392.07 K:
  T = 393.07 K:
  T = 394.07 K:
  T = 395.07 K:
  T = 396.07 K:
  T = 397.07 K:
  T = 398.07 K:
  T = 399.07 K:
  T = 400.07 K:
  T = 401.07 K:
  T = 402.07 K:
  T = 403.07 K:
  T = 404.07 K:
  T = 405.07 K:
  T = 406.07 K:
  T = 407.07 K:
  T = 408.07 K:
  T = 409.07 K:
  T = 410.07 K:
  T = 411.07 K:
  T = 412.07 K:
  T = 413.07 K:
  T = 414.07 K:
  T = 415.07 K:
  T = 416.07 K:
  T = 417.07 K:
  T = 418.07 K:
  T = 419.07 K:
  T = 420.07 K:
  T = 421.07 K:
  T = 422.07 K:
  T = 423.07 K:
  T = 424.07 K:
  T = 425.07 K:
  T = 426.07 K:
  T = 427.07 K:
  T = 428.07 K:
  T = 429.07 K:
  T = 430.07 K:
  T = 431.07 K:
  T = 432.07 K:
  T = 433.07 K:
  T = 434.07 K:
  T = 435.07 K:
  T = 436.07 K:
  T = 437.07 K:
  T = 438.07 K:
  T = 439.07 K:
  T = 440.07 K:
  T = 44

In [21]:

miscibility_results[0]['G_mix']

array([ 0.        , -0.02057467, -0.02789996, -0.03068277, -0.03076577,
       -0.02901905, -0.02595128, -0.02189565, -0.01708657, -0.01169732,
       -0.00586076,  0.00031841,  0.00675586,  0.01338243,  0.02014056,
        0.02698178,  0.03386486,  0.04075436,  0.04761964,  0.05443399,
        0.061174  ,  0.06781906,  0.0743509 ,  0.08075331,  0.08701181,
        0.09311345,  0.09904658,  0.10480074,  0.11036647,  0.11573523,
        0.12089928,  0.1258516 ,  0.13058581,  0.13509612,  0.13937724,
        0.14342437,  0.14723316,  0.15079962,  0.15412015,  0.15719145,
        0.16001058,  0.16257484,  0.16488183,  0.16692938,  0.16871557,
        0.17023871,  0.17149733,  0.17249016,  0.17321614,  0.1736744 ,
        0.17386428,  0.1737853 ,  0.17343719,  0.17281986,  0.17193343,
        0.17077821,  0.16935475,  0.16766379,  0.1657063 ,  0.16348349,
        0.16099684,  0.15824809,  0.15523925,  0.15197267,  0.14845102,
        0.14467733,  0.14065503,  0.13638798,  0.13188053,  0.12